In [1]:
import pandas as pd
import plotly.express as px

df = pd.read_excel("merged_cleaned.xlsx")

print("Shape:", df.shape)
print(
    "Duplicate client-year records:",
    df.duplicated(subset=["CLIENT ID", "YEAR"]).sum()
)

display(df.head())

Shape: (392, 16)
Duplicate client-year records: 0


,CLIENT ID,TYPE,COMMENCEMENT DATE,STAFF STRENGTH,SECTOR,COUNTRY,YEAR,PRESALES AND PARTNERSHIP,TECHNICAL EXPERTISE,PROJECT DELIVERY,POST-SALES SUPPORT,NPS RATING,REVENUE,HARDWARE,SOFTWARE,MANPOWER
0,G1-001,Govt,2020-01-19 00:00:00,> 200,Finance,Singapore,2021,3,4,4,5,8,148180,17321,25562,41201
1,G1-001,Govt,2020-01-19 00:00:00,> 200,Finance,Singapore,2022,5,5,3,4,9,197313,23604,32875,50085
2,G1-001,Govt,2020-01-19 00:00:00,> 200,Finance,Singapore,2023,5,4,3,4,7,349899,51332,71695,89089
3,G1-001,Govt,2020-01-19 00:00:00,> 200,Finance,Singapore,2024,5,5,3,4,8,268821,40969,56452,76545
4,G1-001,Govt,2020-01-19 00:00:00,> 200,Finance,Singapore,2025,5,4,5,5,10,138115,14516,20889,28968


In [2]:
print(
    "Duplicate client-year records:",
    df.duplicated(subset=["CLIENT ID", "YEAR"]).sum()
)

Duplicate client-year records: 0


In [3]:
service_columns = [
    "PRESALES AND PARTNERSHIP",
    "TECHNICAL EXPERTISE",
    "PROJECT DELIVERY",
    "POST-SALES SUPPORT"
]

In [4]:
type_summary = (
    df.groupby("TYPE")[service_columns]
      .mean()
      .round(2)
      .reset_index()
)

display(type_summary)

,TYPE,PRESALES AND PARTNERSHIP,TECHNICAL EXPERTISE,PROJECT DELIVERY,POST-SALES SUPPORT
0,Govt,4.09,3.94,4.35,3.91
1,NPO,3.76,3.88,4.26,4.12
2,Private,3.96,3.92,4.01,3.80


In [5]:
type_long = type_summary.melt(
    id_vars="TYPE",
    var_name="Service Dimension",
    value_name="Average Rating"
)

display(type_long)

,TYPE,Service Dimension,Average Rating
0,Govt,PRESALES AND PARTNERSHIP,4.09
1,NPO,PRESALES AND PARTNERSHIP,3.76
2,Private,PRESALES AND PARTNERSHIP,3.96
3,Govt,TECHNICAL EXPERTISE,3.94
4,NPO,TECHNICAL EXPERTISE,3.88
5,Private,TECHNICAL EXPERTISE,3.92
6,Govt,PROJECT DELIVERY,4.35
7,NPO,PROJECT DELIVERY,4.26
8,Private,PROJECT DELIVERY,4.01
9,Govt,POST-SALES SUPPORT,3.91


In [6]:
fig = px.bar(
    type_long,
    x="TYPE",
    y="Average Rating",
    color="Service Dimension",
    barmode="group",
    category_orders={
        "Service Dimension": service_columns
    }
)

In [7]:
def get_trace_data(data):
    return {
        "x": [
            data.loc[
                data["Service Dimension"] == dimension, "TYPE"
            ].tolist()
            for dimension in service_columns
        ],
        "y": [
            data.loc[
                data["Service Dimension"] == dimension, "Average Rating"
            ].tolist()
            for dimension in service_columns
        ]
    }


buttons = [
    dict(
        label="All Years",
        method="update",
        args=[
            get_trace_data(type_long),
            {"title.text": "Average Service Ratings by Client Type — All Years"}
        ]
    )
]

for year in sorted(yearly_long["YEAR"].unique()):
    selected_year = yearly_long[yearly_long["YEAR"] == year]

    buttons.append(
        dict(
            label=str(year),
            method="update",
            args=[
                get_trace_data(selected_year),
                {"title.text": f"Average Service Ratings by Client Type — {year}"}
            ]
        )
    )

NameError: name 'yearly_long' is not defined

In [ ]:
fig.update_layout(
    updatemenus=[
        dict(
            buttons=buttons,
            direction="down",
            showactive=True,
            x=1,
            xanchor="right",
            y=1.18,
            yanchor="top"
        )
    ],
    template="plotly_white",
    title="Average Service Ratings by Client Type — All Years",
    xaxis_title="Client Type",
    yaxis_title="Average Rating",
    legend_title="Service Dimension"
)

fig.update_yaxes(range=[0, 5.3], dtick=1)

fig.show()

In [8]:
import plotly.graph_objects as go

nps_summary = (
    df.groupby(["YEAR", "TYPE"], as_index=False)["NPS RATING"]
      .mean()
      .round(2)
)

display(nps_summary)

,YEAR,TYPE,NPS RATING
0,2021,Govt,7.67
1,2021,NPO,7.50
2,2021,Private,7.38
3,2022,Govt,7.60
4,2022,NPO,8.33
5,2022,Private,7.68
6,2023,Govt,7.56
7,2023,NPO,7.83
8,2023,Private,7.69
9,2024,Govt,8.00


In [9]:
print(
    nps_summary.groupby("TYPE")["YEAR"].nunique()
)

TYPE
Govt       5
NPO        5
Private    5
Name: YEAR, dtype: int64


In [10]:
fig2 = go.Figure()

for client_type in nps_summary["TYPE"].unique():
    selected_data = nps_summary[
        nps_summary["TYPE"] == client_type
    ]

    fig2.add_trace(
        go.Scatter(
            x=selected_data["YEAR"],
            y=selected_data["NPS RATING"],
            mode="lines+markers",
            name=client_type
        )
    )

In [11]:
fig2.update_layout(
    title="Average NPS Rating by Client Type (2021–2025)",
    xaxis_title="Year",
    yaxis_title="Average NPS Rating",
    template="plotly_white",
    hovermode="x unified"
)

fig2.update_xaxes(
    tickmode="linear",
    dtick=1
)

fig2.update_yaxes(
    range=[0, 10],
    dtick=1
)

fig2.show()

In [12]:
buttons = [
    dict(
        label="All Client Types",
        method="update",
        args=[
            {"visible": [True] * len(fig2.data)},
            {"title.text": "Average NPS Rating — All Client Types"}
        ]
    )
]

for index, trace in enumerate(fig2.data):
    visibility = [False] * len(fig2.data)
    visibility[index] = True

    buttons.append(
        dict(
            label=trace.name,
            method="update",
            args=[
                {"visible": visibility},
                {"title.text": f"Average NPS Rating — {trace.name}"}
            ]
        )
    )

In [13]:
fig2.update_layout(
    updatemenus=[
        dict(
            buttons=buttons,
            direction="down",
            showactive=True,
            x=1,
            xanchor="right",
            y=1.18,
            yanchor="top"
        )
    ]
)

fig2.show()

In [14]:
correlations = (
    df[service_columns + ["NPS RATING"]]
      .corr(method="spearman")["NPS RATING"]
      .drop("NPS RATING")
      .sort_values()
)

display(correlations)

PRESALES AND PARTNERSHIP    0.412219
POST-SALES SUPPORT          0.444440
PROJECT DELIVERY            0.538029
TECHNICAL EXPERTISE         0.543798
Name: NPS RATING, dtype: float64

In [15]:
correlation_df = (
    correlations
    .rename("Correlation with NPS")
    .reset_index()
    .rename(columns={"index": "Service Dimension"})
)

display(correlation_df)

,Service Dimension,Correlation with NPS
0,PRESALES AND PARTNERSHIP,0.412219
1,POST-SALES SUPPORT,0.444440
2,PROJECT DELIVERY,0.538029
3,TECHNICAL EXPERTISE,0.543798


In [16]:
fig3 = go.Figure()

fig3.add_trace(
    go.Bar(
        x=correlation_df["Correlation with NPS"],
        y=correlation_df["Service Dimension"],
        orientation="h",
        text=correlation_df["Correlation with NPS"],
        texttemplate="%{text:.2f}",
        textposition="outside",
        marker=dict(
            color=correlation_df["Correlation with NPS"],
            colorscale="Blues",
            cmin=0,
            cmax=1
        ),
        hovertemplate=(
            "<b>%{y}</b><br>"
            "Spearman correlation: %{x:.3f}"
            "<extra></extra>"
        )
    )
)

In [17]:
fig3.update_layout(
    title="Association Between Service Dimensions and NPS",
    xaxis_title="Spearman Correlation with NPS",
    yaxis_title="Service Dimension",
    template="plotly_white",
    showlegend=False
)

fig3.update_xaxes(
    range=[-1, 1],
    zeroline=True,
    zerolinewidth=2,
    zerolinecolor="black"
)

fig3.show()

In [18]:
client_groups = {
    "All Client Types": df
}

for client_type, group_data in df.groupby("TYPE"):
    client_groups[client_type] = group_data

In [19]:
correlation_tables = {}

for group_name, group_data in client_groups.items():
    group_correlations = (
        group_data[service_columns + ["NPS RATING"]]
        .corr(method="spearman")["NPS RATING"]
        .drop("NPS RATING")
        .sort_values()
        .rename("Correlation with NPS")
        .reset_index()
        .rename(columns={"index": "Service Dimension"})
    )

    correlation_tables[group_name] = group_correlations

In [20]:
for group_name, table in correlation_tables.items():
    print(f"\n{group_name}")
    display(table)


All Client Types


,Service Dimension,Correlation with NPS
0,PRESALES AND PARTNERSHIP,0.412219
1,POST-SALES SUPPORT,0.444440
2,PROJECT DELIVERY,0.538029
3,TECHNICAL EXPERTISE,0.543798



Govt


,Service Dimension,Correlation with NPS
0,PRESALES AND PARTNERSHIP,0.246420
1,POST-SALES SUPPORT,0.426670
2,TECHNICAL EXPERTISE,0.521968
3,PROJECT DELIVERY,0.558538



NPO


,Service Dimension,Correlation with NPS
0,POST-SALES SUPPORT,0.281493
1,TECHNICAL EXPERTISE,0.369406
2,PRESALES AND PARTNERSHIP,0.619117
3,PROJECT DELIVERY,0.669775



Private


,Service Dimension,Correlation with NPS
0,PRESALES AND PARTNERSHIP,0.425568
1,POST-SALES SUPPORT,0.464374
2,PROJECT DELIVERY,0.521487
3,TECHNICAL EXPERTISE,0.571417


In [21]:
fig3 = go.Figure()

group_names = list(correlation_tables.keys())

for index, group_name in enumerate(group_names):
    table = correlation_tables[group_name]

    fig3.add_trace(
        go.Bar(
            x=table["Correlation with NPS"],
            y=table["Service Dimension"],
            orientation="h",
            name=group_name,
            visible=(index == 0),
            text=table["Correlation with NPS"],
            texttemplate="%{text:.2f}",
            textposition="auto",
            marker=dict(
                color=table["Correlation with NPS"],
                colorscale="RdBu",
                cmin=-1,
                cmax=1
            ),
            hovertemplate=(
                "<b>%{y}</b><br>"
                "Spearman correlation: %{x:.3f}"
                "<extra></extra>"
            )
        )
    )

In [22]:
buttons = []

for index, group_name in enumerate(group_names):
    visibility = [False] * len(group_names)
    visibility[index] = True

    buttons.append(
        dict(
            label=group_name,
            method="update",
            args=[
                {"visible": visibility},
                {
                    "title.text":
                    f"Service Dimensions Associated with NPS — {group_name}"
                }
            ]
        )
    )

In [23]:
fig3.update_layout(
    title="Service Dimensions Associated with NPS — All Client Types",
    xaxis_title="Spearman Correlation with NPS",
    yaxis_title="Service Dimension",
    template="plotly_white",
    showlegend=False,

    updatemenus=[
        dict(
            buttons=buttons,
            direction="down",
            showactive=True,
            x=1,
            xanchor="right",
            y=1.18,
            yanchor="top"
        )
    ]
)

fig3.update_xaxes(
    range=[-1, 1],
    zeroline=True,
    zerolinewidth=2,
    zerolinecolor="black"
)

fig3.show()

In [24]:
satisfaction_long = df.melt(
    id_vars=["CLIENT ID", "TYPE", "YEAR"],
    value_vars=service_columns,
    var_name="Service Dimension",
    value_name="Rating"
)

display(satisfaction_long.head())

,CLIENT ID,TYPE,YEAR,Service Dimension,Rating
0,G1-001,Govt,2021,PRESALES AND PARTNERSHIP,3
1,G1-001,Govt,2022,PRESALES AND PARTNERSHIP,5
2,G1-001,Govt,2023,PRESALES AND PARTNERSHIP,5
3,G1-001,Govt,2024,PRESALES AND PARTNERSHIP,5
4,G1-001,Govt,2025,PRESALES AND PARTNERSHIP,5


In [25]:
print("Shape:", satisfaction_long.shape)
print("Missing ratings:", satisfaction_long["Rating"].isna().sum())
print(satisfaction_long["Service Dimension"].value_counts())

Shape: (1568, 5)
Missing ratings: 0
Service Dimension
PRESALES AND PARTNERSHIP    392
TECHNICAL EXPERTISE         392
PROJECT DELIVERY            392
POST-SALES SUPPORT          392
Name: count, dtype: int64


In [26]:
default_dimension = service_columns[0]

box_data = satisfaction_long[
    satisfaction_long["Service Dimension"] == default_dimension
]

In [27]:
fig4 = px.box(
    box_data,
    x="TYPE",
    y="Rating",
    color="TYPE",
    points="outliers",
    labels={
        "TYPE": "Client Type",
        "Rating": "Service Rating"
    },
    title=f"Distribution of {default_dimension.title()} Ratings by Client Type"
)

In [28]:
fig4.update_layout(
    template="plotly_white",
    showlegend=False
)

fig4.update_yaxes(
    range=[0.5, 5.5],
    dtick=1
)

fig4.show()